# anndata

## Example 1: build an AnnData object

In [ ]:
# Example 1: X with obs and var annotations, obsm embeddings and layers
import numpy as np
import pandas as pd
import anndata as ad

X = np.arange(12, dtype=np.float32).reshape(4, 3)
adata = ad.AnnData(X, obs=pd.DataFrame({"group": ["a", "a", "b", "b"]}, index=[f"cell{i}" for i in range(4)]),
                   var=pd.DataFrame(index=["g1", "g2", "g3"]))
adata.obsm["X_pca"] = np.ones((4, 2))
adata.layers["counts"] = X * 2
assert adata.shape == (4, 3) and list(adata.var_names) == ["g1", "g2", "g3"]
print("Example 1:", adata)

## Example 2: subsetting by name and by mask

In [ ]:
# Example 2: views slice X, obs, obsm and layers together
import numpy as np
import anndata as ad

adata = ad.AnnData(np.arange(20, dtype=np.float32).reshape(5, 4))
adata.obs["keep"] = [True, False, True, False, True]
sub = adata[adata.obs["keep"].to_numpy(), ["1", "3"]]
assert sub.is_view and sub.shape == (3, 2)
assert sub.X.tolist() == [[1.0, 3.0], [9.0, 11.0], [17.0, 19.0]], sub.X
print("Example 2:", sub.shape, sub.X.tolist())

## Example 3: .h5ad and Zarr round trips with a sparse X

In [ ]:
# Example 3: write_h5ad / read_h5ad and write_zarr / read_zarr keep a CSR matrix and annotations
import pathlib, tempfile
import numpy as np
import scipy.sparse as sp
import anndata as ad

folder = pathlib.Path(tempfile.mkdtemp())
adata = ad.AnnData(sp.random_array((50, 20), density=0.1, format="csr", rng=0, dtype=np.float32))
adata.obs["batch"] = ["x", "y"] * 25
adata.write_h5ad(folder / "data.h5ad")
adata.write_zarr(folder / "data.zarr")
for back in (ad.read_h5ad(folder / "data.h5ad"), ad.read_zarr(folder / "data.zarr")):
    assert sp.issparse(back.X) and (back.X != adata.X).nnz == 0 and list(back.obs["batch"]) == list(adata.obs["batch"])
print("Example 3:", sorted(p.name for p in folder.iterdir()))

## Example 4: concatenation

In [ ]:
# Example 4: anndata.concat joins along cells, with a label column and outer-joined genes
import numpy as np
import pandas as pd
import anndata as ad

a = ad.AnnData(np.ones((2, 2)), var=pd.DataFrame(index=["g1", "g2"]))
b = ad.AnnData(np.zeros((3, 2)), var=pd.DataFrame(index=["g2", "g3"]))
both = ad.concat({"a": a, "b": b}, label="sample", join="outer", index_unique="-")
assert both.shape == (5, 3) and sorted(both.var_names) == ["g1", "g2", "g3"]
assert both.obs["sample"].value_counts().to_dict() == {"b": 3, "a": 2}
print("Example 4:", both.shape, list(both.obs_names))

## Example 5: a Dask-backed X

In [ ]:
# Example 5: X can be a dask array; operations stay lazy until computed
import dask.array as da
import numpy as np
import anndata as ad

X = da.from_array(np.arange(100, dtype=np.float32).reshape(10, 10), chunks=(5, 5))
adata = ad.AnnData(X)
sub = adata[:4, :3]
assert isinstance(sub.X, da.Array)
assert sub.X.compute().tolist()[0] == [0.0, 1.0, 2.0]
print("Example 5:", type(adata.X).__name__, sub.X.compute().shape)